# init

In [1]:
import os
from multiprocessing import Pool
from pathlib import Path

import h5py
import numpy as np

import gutpython

In [2]:
initial_throwaway_time = 350
data_storage_dir = "/big-data/knappa/gutpython/sims"

In [3]:
%matplotlib ipympl

In [4]:
%load_ext jupyterlab_notify

# define param sampling

In [5]:
default_numerical_param_dict = dict(
    max_stuck_chance=50,
    low_stuck_bound=2,
    unstuck_chance=10.0,
    mid_stuck_conc=10.0,
    seed_chance=5.0,
    seed_percent=5.0,
    absorption=0.0,
    reserve_fraction=0.0,
    bifido_lactate_production=0.005,
    flow_dist=0.28,
    bifido_doub=330,
    desulfo_doub=330,
    bacteroid_doub=330,
    clost_doub=330,
    # initialization constants
    init_num_bifidos=23562,
    init_num_bacteroids=5490,
    init_num_closts=921,
    init_num_desulfos=70,
)

In [6]:
def numerical_param_sample():
    proportions = np.exp(np.log(2) * np.random.randn(len(default_numerical_param_dict)))
    return {
        param_name: (
            int(prop * param_value) if isinstance(param_value, int) else float(prop * param_value)
        )
        for prop, (param_name, param_value) in zip(
            proportions, default_numerical_param_dict.items()
        )
    }

In [7]:
def control_param_sample():
    control_params = {}
    p = 0.1

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_bacteroids"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_bacteroids"] = lambda t: 0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_bifidos"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_bifidos"] = lambda t: 0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_closts"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_closts"] = lambda t: 0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["in_conc_desulfos"] = lambda t: (
            10 if 0 <= t - initial_throwaway_time < 100 else 0
        )
    else:
        control_params["in_conc_desulfos"] = lambda t: 0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["cs_inflow"] = lambda t: (
                0.2 if 0 <= t - initial_throwaway_time < 100 else 0.1
            )
        else:
            control_params["cs_inflow"] = lambda t: (
                0.05 if 0 <= t - initial_throwaway_time < 100 else 0.1
            )
    else:
        control_params["cs_inflow"] = lambda t: 0.1

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["fo_inflow"] = lambda t: (
                50.0 if 0 <= t - initial_throwaway_time < 100 else 25.0
            )
        else:
            control_params["fo_inflow"] = lambda t: (
                12.5 if 0 <= t - initial_throwaway_time < 100 else 25.0
            )
    else:
        control_params["fo_inflow"] = lambda t: 25.0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["glucose_inflow"] = lambda t: (
                60.0 if 0 <= t - initial_throwaway_time < 100 else 30.0
            )
        else:
            control_params["glucose_inflow"] = lambda t: (
                15.0 if 0 <= t - initial_throwaway_time < 100 else 30.0
            )
    else:
        control_params["glucose_inflow"] = lambda t: 30.0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["inulin_inflow"] = lambda t: (
                20.0 if 0 <= t - initial_throwaway_time < 100 else 10.0
            )
        else:
            control_params["inulin_inflow"] = lambda t: (
                5.0 if 0 <= t - initial_throwaway_time < 100 else 10.0
            )
    else:
        control_params["inulin_inflow"] = lambda t: 10.0

    if np.random.random() < p:
        if np.random.random() < 0.5:
            control_params["lactose_inflow"] = lambda t: (
                30.0 if 0 <= t - initial_throwaway_time < 100 else 15.0
            )
        else:
            control_params["lactose_inflow"] = lambda t: (
                7.5 if 0 <= t - initial_throwaway_time < 100 else 15.0
            )
    else:
        control_params["lactose_inflow"] = lambda t: 15.0

    if np.random.random() < p:
        # base amount is zero. what is the correct amount for the impulse?
        control_params["lactate_inflow"] = lambda t: (
            1.0 if 0 <= t - initial_throwaway_time < 100 else 0.0
        )
    else:
        control_params["lactate_inflow"] = lambda t: 0.0

    return control_params

In [8]:
def make_model(numerical_param_dict, control_param_dict):
    model = gutpython.GutPython(
        **numerical_param_dict,
        **control_param_dict,
        tick_in_flow=480,
    )
    model.setup()

    return model

In [9]:
def take_sample(sample_idx):
    model = make_model(numerical_param_sample(), control_param_sample())
    filename = os.path.join(data_storage_dir, f"sample-{str(sample_idx).zfill(4)}.hdf5")

    for t in range(initial_throwaway_time + 500):
        model.go()
        if t >= initial_throwaway_time:
            model.save(filename, write_mode="a")

# Sample/Simulate

In [10]:
# %%notify
# with Pool(12) as p:
#     p.map(take_sample, range(200))

# Parse 

In [11]:
def parse(filename, time_seq, normalize=True):
    agent_vars = dict()
    computed_properties = dict()
    controls = dict()
    control_values = dict()
    init_parameters = dict()
    measurements = dict()
    molecules = dict()
    parameters = dict()
    with h5py.File(filename, "r") as h5file:
        keys = list(h5file[time_seq].keys())
        key_types = [h5file[time_seq][k].attrs["type"] for k in keys]
        GRID_WIDTH = int(h5file[time_seq]["GRID_WIDTH"][()])
        GRID_HEIGHT = int(h5file[time_seq]["GRID_HEIGHT"][()])

        for key, key_type in zip(keys, key_types):
            target_dict = {
                "agent": agent_vars,
                "computed_property": computed_properties,
                "control": controls,
                "control_value": control_values,
                "init_parameter": init_parameters,
                "measurement": measurements,
                "molecule": molecules,
                "parameter": parameters,
            }[key_type]
            target_dict[key] = h5file[time_seq][key][()]

    macrostate = np.array(
        [
            *[measurements[k] for k in sorted(measurements.keys())],
            *[parameters[k] for k in sorted(parameters.keys())],
            *np.concat(
                [np.atleast_1d(computed_properties[k]) for k in sorted(computed_properties.keys())],
                axis=0,
            ),
            *[control_values[k] for k in sorted(control_values.keys())],
        ],
        dtype=np.float32,
    )

    # shape: (loc[0], loc[1], cell_type, seed, stuck, categorical_age, categorical_energy)
    agent_age_energy_count = np.zeros((GRID_WIDTH, GRID_HEIGHT, 4, 2, 2, 2, 4), dtype=np.int64)
    for cell_type_idx, cell_type in enumerate(["bacteroid", "bifido", "clost", "desulfo"]):

        for loc, age, energy, is_seed, is_stuck in zip(
            agent_vars[f"{cell_type}_locations"].astype(np.int64),
            agent_vars[f"{cell_type}_age"],
            agent_vars[f"{cell_type}_energy"],
            agent_vars[f"{cell_type}_is_seed"],
            agent_vars[f"{cell_type}_is_stuck"],
        ):
            if age < parameters[f"{cell_type}_doub"]:
                categorical_age = 0
            else:
                categorical_age = 1

            if energy < 25:
                categorical_energy = 0
            elif energy < 50:
                categorical_energy = 1
            elif energy < 80:
                categorical_energy = 2
            else:
                categorical_energy = 3

            agent_age_energy_count[
                loc[0],
                loc[1],
                cell_type_idx,
                int(is_seed),
                int(is_stuck),
                categorical_age,
                categorical_energy,
            ] += 1

    # compute the true counts
    counts = np.sum(agent_age_energy_count, axis=(-1, -2))
    # compute 'smoothed' distribution (perturb away any zero counts)
    agent_age_energy_count += 1
    agent_age_energy_dist = (
        agent_age_energy_count
        / np.sum(agent_age_energy_count, axis=(-1, -2))[:, :, :, :, :, np.newaxis, np.newaxis]
    )

    agent_counts = np.squeeze(counts)
    agent_age_energy_dist = np.squeeze(agent_age_energy_dist)

    molecular_microstate = np.squeeze(
        np.array(
            [
                *[molecules[k] for k in sorted(molecules.keys())],
            ],
            dtype=np.float32,
        ).T
    )

    return agent_counts, agent_age_energy_dist, molecular_microstate, macrostate

In [12]:
microstate_files = sorted(
    [
        f
        for f in Path(data_storage_dir).iterdir()
        if str(f.name).endswith(".hdf5") and str(f.name).startswith("sample-")
    ]
)

microstates = list()
for microstate_file in microstate_files:
    with h5py.File(microstate_file, "r") as h5file:
        for k in h5file.keys():
            microstates.append((microstate_file, k))

microstates = sorted(microstates)


def get_sample_number(file_name):
    file_name = str(file_name)
    start = file_name.rfind("-") + 1
    end = file_name.rfind(".")
    return int(file_name[start:end])

In [13]:
def write_sample(x):
    microstate_filename, time_idx = x
    sample_number = get_sample_number(microstate_filename)

    agent_counts, agent_age_energy_dist, molecular_microstate, macrostate = parse(
        microstate_filename, time_idx
    )

    with h5py.File(
        os.path.join(
            microstate_dir,
            f"microstate-{str(sample_number).zfill(4)}-{str(time_idx).zfill(4)}.hdf5",
        ),
        "w",
    ) as h5file:
        h5file.create_dataset(
            "agent_counts",
            shape=agent_counts.shape,
            dtype=np.int32,
            data=agent_counts,
            compression="gzip",
            compression_opts=9,
        )
        h5file.create_dataset(
            "agent_age_energy_dist",
            shape=agent_age_energy_dist.shape,
            dtype=np.float32,
            data=agent_age_energy_dist,
            compression="gzip",
            compression_opts=9,
        )
        h5file.create_dataset(
            "molecular_microstate",
            shape=molecular_microstate.shape,
            dtype=np.float32,
            data=molecular_microstate,
            compression="gzip",
            compression_opts=9,
        )
        h5file.create_dataset(
            "macrostate",
            shape=macrostate.shape,
            dtype=np.float32,
            data=macrostate,
            compression="gzip",
            compression_opts=9,
        )

In [14]:
microstate_dir = os.path.join(data_storage_dir, f"microstates")
try:
    os.mkdir(microstate_dir)
except FileExistsError:
    pass

In [15]:
%%notify
with Pool(12) as p:
    p.map(write_sample, microstates)